In [26]:
import tensorflow as tf
from tensorflow.keras.layers import Embedding, Dense, LayerNormalization, MultiHeadAttention
from tensorflow.keras.models import Model
import numpy as np
import faiss

In [27]:
!pip install faiss-cpu

Let's scrap some data from wikipedia post

In [28]:
# --------------------------
# Sample training text
# --------------------------



def create_wiki_url(topic):
    topic = topic.strip().replace(" ", "_")
    return f"https://en.wikipedia.org/wiki/{topic}"

topics = [
    "Artificial intelligence",
    "Machine learning",
    "Transformer (deep learning)",
    "Retrieval-augmented generation"
]

urls = [create_wiki_url(t) for t in topics]

for url in urls:
    print(url)

https://en.wikipedia.org/wiki/Artificial_intelligence
https://en.wikipedia.org/wiki/Machine_learning
https://en.wikipedia.org/wiki/Transformer_(deep_learning)
https://en.wikipedia.org/wiki/Retrieval-augmented_generation


In [29]:
import requests
from bs4 import BeautifulSoup

def get_text_from_url(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)

    soup = BeautifulSoup(response.text, "html.parser")

    # remove junk
    for tag in soup(["script", "style", "sup", "table"]):
        tag.decompose()

    content = soup.find("div", {"class": "mw-parser-output"})

    text = []
    for p in content.find_all(["p", "h2", "h3"]):
        t = p.get_text(strip=True)
        if t:
            text.append(t)

    return "\n".join(text)
wiki_data = {}

for topic, url in zip(topics, urls):
    print(f"Fetching: {topic}")

    text = get_text_from_url(url)

    if text:
        wiki_data[topic] = text

print("✅ Data collected:", list(wiki_data.keys()))

Fetching: Artificial intelligence
Fetching: Machine learning
Fetching: Transformer (deep learning)
Fetching: Retrieval-augmented generation
✅ Data collected: ['Machine learning', 'Transformer (deep learning)', 'Retrieval-augmented generation']


In [30]:
merged_text = ""

for url in urls:
    print(f"Fetching: {url}")
    merged_text += get_text_from_url(url) + "\n\n"

Fetching: https://en.wikipedia.org/wiki/Artificial_intelligence
Fetching: https://en.wikipedia.org/wiki/Machine_learning
Fetching: https://en.wikipedia.org/wiki/Transformer_(deep_learning)
Fetching: https://en.wikipedia.org/wiki/Retrieval-augmented_generation


In [31]:
print(merged_text[:2000])  # preview



Machine learning(ML) is afield of studyinartificial intelligenceconcerned with the development and study ofstatistical algorithmsthat can learn fromdataandgeneralizeto unseen data, and thus performtaskswithout explicit programming languageinstructions.Within a subdiscipline of machine learning, advances in the field ofdeep learninghave allowedneural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.
Statisticsandmathematical optimisation(mathematical programming) methods compose the foundations of machine learning.Data miningis a related field of study, focusing onexploratory data analysis(EDA) throughunsupervised learning.
From a theoretical viewpoint,probably approximately correct learningprovides a mathematical and statistical framework for describing machine learning. Most traditional machine learning and deep learning algorithms can be described asempirical risk minimisationunder this framework.
History
The termmachi

In [32]:
size= len(merged_text)
print(size)

122717


In [33]:
# --------------------------
# Tokenization
# --------------------------

tokens = merged_text.lower().split()
vocab = sorted(set(tokens))

word2id = {w:i for i,w in enumerate(vocab)}
id2word = {i:w for w,i in word2id.items()}

encoded = [word2id[w] for w in tokens]



In [34]:
# --------------------------
# Create training sequences
# --------------------------

seq_len = 20

X = []
Y = []

for i in range(len(encoded) - seq_len):
    X.append(encoded[i:i+seq_len])
    Y.append(encoded[i+1:i+seq_len+1])

X = np.array(X)
Y = np.array(Y)

indices = np.arange(len(X))
np.random.shuffle(indices)

X = X[indices]
Y = Y[indices]

# --------------------------
# Train / Validation Split
# --------------------------

split = int(0.8 * len(X))

X_train = X[:split]
Y_train = Y[:split]

X_valid = X[split:]
Y_valid = Y[split:]

vocab_size = len(vocab)

embed_dim = 128
heads = 4
ff_dim = 256
max_len = 100


In [35]:
# --------------------------
# Positional Embedding
# --------------------------

class PositionalEmbedding(tf.keras.layers.Layer):

    def __init__(self, vocab_size, embed_dim, max_len):
        super().__init__()

        self.token_emb = Embedding(vocab_size, embed_dim)
        self.pos_emb = Embedding(max_len, embed_dim)

    def call(self, x):

        length = tf.shape(x)[1]

        positions = tf.range(start=0, limit=length, delta=1)
        positions = self.pos_emb(positions)

        x = self.token_emb(x)

        return x + positions

In [36]:
# --------------------------
# GPT Block
# --------------------------

class GPTBlock(tf.keras.layers.Layer):

    def __init__(self, embed_dim, heads, ff_dim):
        super().__init__()

        self.att = MultiHeadAttention(
            num_heads=heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim)
        ])

        self.norm1 = LayerNormalization()
        self.norm2 = LayerNormalization()

    def call(self, x):

        attn = self.att(
            x,
            x,
            use_causal_mask=True
        )

        x = self.norm1(x + attn)

        ffn = self.ffn(x)

        x = self.norm2(x + ffn)

        return x

In [37]:
# --------------------------
# GPT Model
# --------------------------

inputs = tf.keras.Input(shape=(seq_len,))

x = PositionalEmbedding(vocab_size, embed_dim, max_len)(inputs)

x = GPTBlock(embed_dim, heads, ff_dim)(x)
x = GPTBlock(embed_dim, heads, ff_dim)(x)

In [38]:
outputs = Dense(vocab_size, activation="softmax")(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(0.0003),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_embedding            │ (None, 20, 128)        │       644,992 │
│ (PositionalEmbedding)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gpt_block (GPTBlock)            │ (None, 20, 128)        │       330,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gpt_block_1 (GPTBlock)          │ (None, 20, 128)        │       330,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 20, 4939)       │       637,131 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,942,603 (7.41 MB)

 Trainable params: 1,942,603 (7.41 MB)

 Non-trainable params: 0 (0.00 B)

In [39]:
# --------------------------
# Train
# --------------------------
model.fit(
    X_train,
    Y_train,
    validation_data=(X_valid, Y_valid),
    epochs=20,
    batch_size=32
)

Epoch 1/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 25s 25ms/step - accuracy: 0.1663 - loss: 6.2114 - val_accuracy: 0.3440 - val_loss: 4.6311
Epoch 2/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.4511 - loss: 3.5030 - val_accuracy: 0.5361 - val_loss: 2.5949
Epoch 3/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.6749 - loss: 1.8022 - val_accuracy: 0.7821 - val_loss: 1.2942
Epoch 4/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8744 - loss: 0.8742 - val_accuracy: 0.8995 - val_loss: 0.6894
Epoch 5/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9280 - loss: 0.4823 - val_accuracy: 0.9251 - val_loss: 0.4588
Epoch 6/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9383 - loss: 0.3433 - val_accuracy: 0.9292 - val_loss: 0.3849
Epoch 7/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9425 - loss: 0.2879 - val_accuracy: 0.9312 - val_loss: 0.3590
Epoch 8/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9445 - loss: 0.2595 - val_accuracy: 

In [40]:
def chunk_text(text, chunk_size=200):
    words = text.split()
    return [
        " ".join(words[i:i+chunk_size])
        for i in range(0, len(words), chunk_size)
    ]

chunks = []

for topic, text in wiki_data.items():   # from scraping step
    for chunk in chunk_text(text):
        chunks.append(chunk)
print("✅ Total chunks:", len(chunks))

✅ Total chunks: 82


In [41]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# convert text → vectors
doc_embeddings = embedder.encode(chunks)

# FAISS index
dim = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(np.array(doc_embeddings))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [42]:
def retrieve(query, k=2):
    q_emb = embedder.encode([query])

    distances, indices = index.search(q_emb, k)

    return [chunks[i] for i in indices[0]]

In [43]:
def generate_with_rag(query):

    # 🔍 retrieve relevant context
    context_chunks = retrieve(query, k=2)

    context_text = " ".join(context_chunks)

    # 🔥 build prompt
    prompt = context_text + " " + query

    # ⚠️ limit length (important for your small model)
    prompt_words = prompt.split()[-20:]   # keep last tokens
    prompt = " ".join(prompt_words)

    # 🧠 generate using your model
    return generate(prompt)

In [49]:
def generate(seed, steps=200, temperature=0.7):

    words = seed.lower().split()

    for _ in range(steps):

        seq = [word2id.get(w, 0) for w in words[-seq_len:]]

        if len(seq) < seq_len:
            seq = [0] * (seq_len - len(seq)) + seq

        seq = np.array(seq).reshape(1, seq_len)

        pred = model.predict(seq, verbose=0)

        # 🔥 get probabilities for last token
        probs = pred[0, -1]

        # 🔥 temperature scaling
        probs = np.log(probs + 1e-9) / temperature
        probs = np.exp(probs)
        probs = probs / np.sum(probs)

        # 🔥 sampling instead of argmax
        next_id = np.random.choice(len(probs), p=probs)

        next_word = id2word[next_id]

        words.append(next_word)

    return " ".join(words)

In [53]:
print(generate("Transformer"))

transformer about ai. language models learned from data have been shown to contain human-like biases.because human languages contain biases, machines trained on languagecorporawill necessarily also learn these biases.in 2016, microsoft testedtay, achatbotthat learned from twitter, and it quickly picked up racist and sexist language. in an experiment carried out bypropublica, aninvestigative journalismorganisation, a machine learning algorithm's insight into the recidivism rates among prisoners falsely flagged "black defendants high risk twice as often as white defendants".in 2015, google photos once tagged a couple of black people as gorillas, which caused controversy. the gorilla label was subsequently removed, and in 2023, it still cannot recognise gorillas.similar issues with recognising non-white people have been found in many other systems. because of such challenges, the effective use of machine learning may take longer to be adopted in other domains.concern forfairnessin machine